# Saxo Institutional Dashboard (Presentation)

Production-safe, read-only institutional monitoring notebook.
Data is sourced from live dashboard tables and runtime signal/trade history only.


In [ ]:
%pip install -q joblib

import json
from datetime import timedelta

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 250)
pd.set_option('display.float_format', '{:,.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

try:
    display
except NameError:
    def display(obj):
        print(obj)

CATALOG = 'pawan_quant_research_1'
SCHEMA = 'v4'

FEED_TABLE = f'{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_feed'
HEALTH_TABLE = f'{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_health_rollup'
PERF_TABLE = f'{CATALOG}.{SCHEMA}.gold_rt_saxo_dashboard_performance_rollup'
REGIME_SIGNAL_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_regime_signal'
SILVER_RT_TABLE = f'{CATALOG}.{SCHEMA}.silver_rt_1h'
TRADE_LOG_TABLE = f'{CATALOG}.{SCHEMA}.bronze_rt_saxo_trade_log'
MODEL_REGISTRY_TABLE = f'{CATALOG}.{SCHEMA}.model_registry_current'

LIVE_MODEL_PATH = f'/Volumes/{CATALOG}/{SCHEMA}/market_data/models/bt_fred_1d_xgboost_regime_live.joblib'
LOOKBACK_HOURS = 360
TOP_FEATURES = 8
FEATURE_PLOT_DAYS = 30

FEATURES = [
    'CPIAUCSL_z',
    'PCEPI_z',
    'PCE_z',
    'PAYEMS_z',
    'UNRATE_z',
    'GDPC1_z',
    'INDPRO_z',
    'UMCSENT_z',
    'VIXCLS_z',
    'vix3m_z',
    'VIX_Inversion_Z',
    'Price_Mom_24H',
    'CPI_Event_Active',
    'NFP_Event_Active',
]

FEATURE_LABELS = {
    'CPIAUCSL_z': 'CPI',
    'PCEPI_z': 'Core PCE',
    'PCE_z': 'PCE',
    'PAYEMS_z': 'Payrolls',
    'UNRATE_z': 'Unemployment',
    'GDPC1_z': 'GDP',
    'INDPRO_z': 'Industrial Production',
    'UMCSENT_z': 'Sentiment',
    'VIXCLS_z': 'VIX Spot',
    'vix3m_z': 'VIX 3M',
    'VIX_Inversion_Z': 'VIX Inversion',
    'Price_Mom_24H': '24H Price Momentum',
    'CPI_Event_Active': 'CPI Event',
    'NFP_Event_Active': 'NFP Event',
}

REGIME_COLOR = {'Expansion': '#2E8B57', 'Slowdown': '#D4A017', 'Recession': '#C0392B'}


def _to_float(value):
    if value is None:
        return np.nan
    if isinstance(value, pd.Series):
        value = value.dropna()
        return float(value.iloc[0]) if not value.empty else np.nan
    if isinstance(value, (int, float, np.number)):
        return float(value) if value == value else np.nan
    try:
        parsed = float(value)
        return parsed if parsed == parsed else np.nan
    except Exception:
        return np.nan


def _is_missing(value):
    if value is None:
        return True
    try:
        missing = pd.isna(value)
    except Exception:
        return False
    if isinstance(missing, (bool, int, float)):
        return bool(missing)
    return False


def _as_text(value, default='N/A'):
    return default if _is_missing(value) else str(value)


def fmt_money(value):
    value = _to_float(value)
    return 'N/A' if pd.isna(value) else f'${value:,.2f}'


def fmt_pct(value):
    value = _to_float(value)
    return 'N/A' if pd.isna(value) else f'{value:+,.2%}'


def fmt_minutes(value):
    value = _to_float(value)
    return 'N/A' if pd.isna(value) else f'{value:.0f}'


def fmt_timestamp(value):
    ts = pd.to_datetime(value, errors='coerce')
    return 'N/A' if pd.isna(ts) else ts.strftime('%Y-%m-%d %H:%M:%S')


def table_exists(name):
    return spark.catalog.tableExists(name)


def query_to_pandas(sql, empty_columns=None):
    try:
        return spark.sql(sql).toPandas()
    except Exception as exc:
        print(f'Query failed: {exc}')
        return pd.DataFrame(columns=empty_columns or [])


def latest_row(table_name, order_col):
    if not table_exists(table_name):
        return pd.Series(dtype='object')
    rows = query_to_pandas(f'SELECT * FROM {table_name} ORDER BY {order_col} DESC LIMIT 1')
    return rows.iloc[0] if not rows.empty else pd.Series(dtype='object')


def latest_rows(table_name, order_col, limit=LOOKBACK_HOURS):
    if not table_exists(table_name):
        return pd.DataFrame()
    q = f'''
        SELECT *
        FROM (
            SELECT *
            FROM {table_name}
            ORDER BY {order_col} DESC
            LIMIT {int(limit)}
        ) t
        ORDER BY {order_col} ASC
    '''
    return query_to_pandas(q)


def table_columns(table_name):
    schema = query_to_pandas(f'DESCRIBE {table_name}', empty_columns=['col_name'])
    if schema.empty or 'col_name' not in schema.columns:
        return []
    return [str(c).strip() for c in schema['col_name'].tolist()]


def load_model_metadata():
    default_version = 'bootstrap_live_model'
    default_path = LIVE_MODEL_PATH
    if table_exists(MODEL_REGISTRY_TABLE):
        rows = query_to_pandas(
            f'''
                SELECT model_version, model_path
                FROM {MODEL_REGISTRY_TABLE}
                ORDER BY promoted_ts DESC
                LIMIT 1
            '''
        )
        if not rows.empty and not _is_missing(rows.iloc[0].get('model_path')):
            return _as_text(rows.iloc[0].get('model_version'), default_version), rows.iloc[0].get('model_path')
    return default_version, default_path


def load_model():
    model_version, model_path = load_model_metadata()
    try:
        model = joblib.load(model_path)
        return model, model_version, model_path
    except Exception as exc:
        print(f'Model load skipped ({model_path}): {exc}')
        return None, model_version, model_path


def feature_importance(model):
    if model is not None and hasattr(model, 'feature_importances_'):
        importance = np.array(model.feature_importances_, dtype=float)
    else:
        importance = np.ones(len(FEATURES), dtype=float)

    if importance.ndim != 1 or importance.size != len(FEATURES) or np.isclose(np.nan_to_num(importance).sum(), 0):
        importance = np.ones(len(FEATURES), dtype=float)

    frame = pd.DataFrame({'feature': FEATURES, 'importance': importance})
    frame = frame.sort_values('importance', ascending=False).reset_index(drop=True)
    frame['importance_pct'] = frame['importance'] / frame['importance'].sum()
    frame['label'] = frame['feature'].map(FEATURE_LABELS)
    return frame


def top_prob_from_row(row):
    columns = {"pct_expansion": 'Expansion', 'pct_slowdown': 'Slowdown', 'pct_recession': 'Recession'}
    best_label = 'N/A'
    best_value = np.nan
    for c, label in columns.items():
        val = _to_float(row.get(c))
        if pd.notna(val) and (pd.isna(best_value) or val > best_value):
            best_label = label
            best_value = val
    return best_label, best_value


def detect_table_columns(frame, candidates):
    cols = set(frame.columns)
    return [c for c in candidates if c in cols]


latest_feed = latest_row(FEED_TABLE, 'feed_ts')
latest_health = latest_row(HEALTH_TABLE, 'rollup_ts')
latest_perf = latest_row(PERF_TABLE, 'rollup_ts')
execution_mode = _as_text(latest_feed.get('execution_mode', ''), '') if hasattr(latest_feed, 'get') else ''
if not execution_mode:
    execution_mode = _as_text(latest_perf.get('execution_mode', ''), '')
safe_execution_mode = execution_mode.replace("'", "''") if execution_mode else ''
mode_filter = '' if not safe_execution_mode else f"WHERE execution_mode = '{safe_execution_mode}'"

perf_history = query_to_pandas(f"""
    SELECT *
    FROM {PERF_TABLE}
    {mode_filter}
    ORDER BY rollup_ts DESC
    LIMIT {LOOKBACK_HOURS}
""")

health_history = query_to_pandas(f"""
    SELECT *
    FROM {HEALTH_TABLE}
    {mode_filter}
    ORDER BY rollup_ts DESC
    LIMIT {LOOKBACK_HOURS}
""")

regime_history = query_to_pandas(f"""
    SELECT feed_ts AS regime_ts,
           COALESCE(regime_name, CAST(regime_int AS STRING)) AS regime_name,
           CAST(regime_int AS INT) AS regime_int,
           pct_expansion, pct_slowdown, pct_recession
    FROM {FEED_TABLE}
    {mode_filter}
    ORDER BY feed_ts DESC
    LIMIT {LOOKBACK_HOURS}
""")

signal_history = query_to_pandas(f"""
    SELECT *
    FROM (
      SELECT timestamp, regime_name, regime_int, position
      FROM {REGIME_SIGNAL_TABLE}
      ORDER BY timestamp DESC
      LIMIT {LOOKBACK_HOURS}
    ) t
    ORDER BY timestamp ASC
""")

silver_cols = table_columns(SILVER_RT_TABLE)
if silver_cols:
    requested = [f for f in FEATURES if f in silver_cols]
    query_cols = ', '.join(['timestamp'] + requested)
    feature_history = query_to_pandas(f"SELECT {query_cols} FROM {SILVER_RT_TABLE} ORDER BY timestamp DESC LIMIT {LOOKBACK_HOURS}")
else:
    feature_history = pd.DataFrame(columns=['timestamp'] + FEATURES)

if 'timestamp' in feature_history.columns:
    feature_history['timestamp'] = pd.to_datetime(feature_history['timestamp'], errors='coerce')
    feature_history = feature_history.sort_values('timestamp').reset_index(drop=True)
    for col in [c for c in feature_history.columns if c != 'timestamp']:
        feature_history[col] = pd.to_numeric(feature_history[col], errors='coerce')

trade_history = query_to_pandas(f"SELECT * FROM {TRADE_LOG_TABLE} ORDER BY trade_ts DESC LIMIT 250", empty_columns=['trade_ts'])
if 'trade_ts' in trade_history.columns:
    trade_history['trade_ts'] = pd.to_datetime(trade_history['trade_ts'], errors='coerce')
    trade_history = trade_history.sort_values('trade_ts').reset_index(drop=True)

model, model_version, model_path = load_model()
importance_frame = feature_importance(model)
importance_map = dict(zip(importance_frame['feature'], importance_frame['importance_pct']))

perf_history = perf_history.sort_values('rollup_ts').reset_index(drop=True) if not perf_history.empty else perf_history
if not perf_history.empty:
    perf_history['rollup_ts'] = pd.to_datetime(perf_history['rollup_ts'], errors='coerce')
    for c in ['account_equity', 'account_cash', 'drawdown_pct', 'regime_drawdown_pct', 'alpha_pct', 'strategy_return_1h', 'alpha_1h', 'strategy_return_1d', 'alpha_1d']:
        if c in perf_history.columns:
            perf_history[c] = pd.to_numeric(perf_history[c], errors='coerce')

if not health_history.empty:
    health_history['rollup_ts'] = pd.to_datetime(health_history['rollup_ts'], errors='coerce')
    health_history = health_history.sort_values('rollup_ts').reset_index(drop=True)

if not regime_history.empty:
    regime_history['regime_ts'] = pd.to_datetime(regime_history['regime_ts'], errors='coerce')
    regime_history['pct_expansion'] = pd.to_numeric(regime_history.get('pct_expansion'), errors='coerce')
    regime_history['pct_slowdown'] = pd.to_numeric(regime_history.get('pct_slowdown'), errors='coerce')
    regime_history['pct_recession'] = pd.to_numeric(regime_history.get('pct_recession'), errors='coerce')
    regime_history = regime_history.sort_values('regime_ts').reset_index(drop=True)

if not signal_history.empty:
    signal_history['timestamp'] = pd.to_datetime(signal_history['timestamp'], errors='coerce')
    signal_history = signal_history.sort_values('timestamp').reset_index(drop=True)
    signal_history['regime_int'] = pd.to_numeric(signal_history['regime_int'], errors='coerce')


displayHTML('<h2>1. Executive decision header</h2>')
prob_label, prob_value = top_prob_from_row(latest_feed)
kpi = [
    ('Current regime', _as_text(latest_feed.get('regime_name', 'N/A'))),
    ('Regime confidence', f'{prob_label}: {fmt_pct(prob_value)}' if pd.notna(_to_float(prob_value)) else 'N/A'),
    ('Current position', _as_text(latest_feed.get('actual_ticker', latest_feed.get('position', 'N/A')))),
    ('Execution mode', _as_text(latest_feed.get('execution_mode', latest_perf.get('execution_mode', 'N/A')))),
    ('Account equity', fmt_money(latest_feed.get('account_equity'))),
    ('Account cash', fmt_money(latest_feed.get('account_cash'))),
    ('Account balance', fmt_money(latest_feed.get('account_balance'))),
    ('1h strategy', fmt_pct(latest_perf.get('strategy_return_1h'))),
    ('1h alpha', fmt_pct(latest_perf.get('alpha_1h'))),
    ('1d strategy', fmt_pct(latest_perf.get('strategy_return_1d'))),
    ('1d alpha', fmt_pct(latest_perf.get('alpha_1d'))),
    ('Current drawdown', fmt_pct(latest_feed.get('drawdown_pct'))),
    ('Freshness (min)', fmt_minutes(latest_feed.get('dashboard_data_freshness_minutes'))),
    ('Latest audit timestamp', fmt_timestamp(latest_feed.get('latest_audit_ts'))),
    ('Last execution action', _as_text(latest_feed.get('last_execution_action'))),
    ('Skip reason', _as_text(latest_feed.get('last_skip_reason'))),
    ('Error reason', _as_text(latest_feed.get('last_error_reason'))),
]
display(pd.DataFrame(kpi, columns=['Metric', 'Value']))


displayHTML('<h2>2. Strategy vs SPY performance</h2>')
if perf_history.empty:
    print('No strategy performance history is available yet.')
else:
    aligned = perf_history.copy()
    aligned = aligned.dropna(subset=['rollup_ts', 'account_equity'])
    if aligned.empty:
        print('No aligned performance records available.')
    else:
        aligned['strategy_return'] = aligned['account_equity'] / aligned['account_equity'].iloc[0] - 1.0
        if 'account_cash' in aligned.columns and aligned['account_cash'].notna().all():
            aligned['cash_return'] = aligned['account_cash'] / aligned['account_cash'].iloc[0] - 1.0

        spy_df = pd.DataFrame()
        if 'spy_close' in trade_history.columns:
            spy_df = trade_history[['trade_ts', 'spy_close']].copy() if 'trade_ts' in trade_history.columns else pd.DataFrame()
            spy_df = spy_df.dropna()

        if not spy_df.empty:
            spy_df = spy_df.sort_values('trade_ts')
            aligned = pd.merge_asof(
                aligned.sort_values('rollup_ts'),
                spy_df.sort_values('trade_ts'),
                left_on='rollup_ts',
                right_on='trade_ts',
                direction='backward'
            )
            aligned['spy_close'] = pd.to_numeric(aligned['spy_close'], errors='coerce')
            if aligned['spy_close'].notna().any():
                first_spy = float(aligned['spy_close'].dropna().iloc[0])
                aligned['spy_return'] = aligned['spy_close'] / first_spy - 1.0

        fig = plt.figure(figsize=(16, 10))
        gs = fig.add_gridspec(3, 1, height_ratios=[3.2, 1.2, 1.0], hspace=0.25)
        ax_perf = fig.add_subplot(gs[0, 0])
        ax_alpha = fig.add_subplot(gs[1, 0], sharex=ax_perf)
        ax_dd = fig.add_subplot(gs[2, 0], sharex=ax_perf)

        if not regime_history.empty:
            window_start = aligned['rollup_ts'].iloc[0]
            window_end = aligned['rollup_ts'].iloc[-1]
            for idx, row in regime_history.iterrows():
                start_t = row['regime_ts']
                end_t = window_end if idx + 1 >= len(regime_history) else regime_history.loc[idx + 1, 'regime_ts']
                if pd.isna(start_t) or pd.isna(end_t) or end_t <= start_t:
                    continue
                if start_t < window_start:
                    start_t = window_start
                color = REGIME_COLOR.get(_as_text(row['regime_name'], 'Unknown'), '#9E9E9E')
                ax_perf.axvspan(start_t, min(end_t, window_end), color=color, alpha=0.15)

        ax_perf.plot(aligned['rollup_ts'], aligned['strategy_return'] * 100, linewidth=2.2, label='Strategy')
        if 'spy_return' in aligned.columns:
            ax_perf.plot(aligned['rollup_ts'], aligned['spy_return'] * 100, linewidth=1.5, linestyle='--', label='SPY proxy', alpha=0.8)
        ax_perf.set_title('Saxo strategy vs SPY')
        ax_perf.set_ylabel('Return (%)')
        ax_perf.grid(alpha=0.3)
        ax_perf.legend(loc='upper left')

        ax_alpha.plot(aligned['rollup_ts'], pd.to_numeric(aligned.get('alpha_pct', np.nan), errors='coerce') * 100, linewidth=2.0, label='Alpha')
        ax_alpha.plot(aligned['rollup_ts'], pd.to_numeric(aligned.get('alpha_1h', np.nan), errors='coerce') * 100, linewidth=1.3, linestyle=':', label='Alpha 1h')
        ax_alpha.plot(aligned['rollup_ts'], pd.to_numeric(aligned.get('alpha_1d', np.nan), errors='coerce') * 100, linewidth=1.3, linestyle='--', label='Alpha 1d')
        ax_alpha.axhline(0, color='black', linestyle='--', alpha=0.5)
        ax_alpha.set_ylabel('Alpha (%)')
        ax_alpha.set_title('Alpha panel')
        ax_alpha.grid(alpha=0.3)
        ax_alpha.legend(loc='upper left')

        drawdown_series = pd.to_numeric(aligned.get('drawdown_pct', np.nan), errors='coerce').fillna(0)
        ax_dd.plot(aligned['rollup_ts'], drawdown_series * 100, linewidth=1.8, color='#B03A2E', label='Strategy drawdown')
        if 'regime_drawdown_pct' in aligned.columns:
            ax_dd.plot(aligned['rollup_ts'], pd.to_numeric(aligned.get('regime_drawdown_pct', np.nan), errors='coerce') * 100, linewidth=1.3, color='#D35400', label='Regime drawdown')
        ax_dd.axhline(0, color='black', linestyle='--', alpha=0.5)
        ax_dd.set_ylabel('Drawdown (%)')
        ax_dd.set_title('Drawdown panel')
        ax_dd.grid(alpha=0.3)
        ax_dd.legend(loc='lower right')
        plt.tight_layout()
        plt.show()


displayHTML('<h2>3. Historical regime / signal monitoring</h2>')
if not regime_history.empty:
    regime_head = regime_history[['regime_ts', 'regime_name', 'regime_int', 'pct_expansion', 'pct_slowdown', 'pct_recession']].copy()
    regime_head['regime_ts'] = regime_head['regime_ts'].dt.strftime('%Y-%m-%d %H:%M')
    regime_head = regime_head.sort_values('regime_ts', ascending=False).head(24)
    regime_head['pct_expansion'] = regime_head['pct_expansion'].apply(fmt_pct)
    regime_head['pct_slowdown'] = regime_head['pct_slowdown'].apply(fmt_pct)
    regime_head['pct_recession'] = regime_head['pct_recession'].apply(fmt_pct)
    regime_head.columns = ['Timestamp', 'Regime', 'Regime code', 'Expansion', 'Slowdown', 'Recession']
    display(regime_head)

    fig2, (axr1, axr2) = plt.subplots(2, 1, figsize=(16, 7), sharex=True, gridspec_kw={'height_ratios':[2,1]})
    prob_cols = [c for c in ['pct_expansion', 'pct_slowdown', 'pct_recession'] if regime_history[c].notna().any()]
    if prob_cols:
        for c in prob_cols:
            axr1.plot(regime_history['regime_ts'], regime_history[c] * 100, linewidth=1.6, label=c.replace('pct_', '').replace('_', ' ').title())
        axr1.set_title('Regime probability tracker')
        axr1.set_ylabel('Probability (%)')
        axr1.set_ylim(0, 100)
        axr1.grid(alpha=0.3)
        axr1.legend(loc='upper right')
    else:
        axr1.axis('off')
        axr1.text(0.5, 0.5, 'No regime probability values in current feed window.', ha='center', va='center')

    if not signal_history.empty:
        sig_plot = signal_history[['timestamp', 'regime_int']].dropna()
        sig_plot['regime_step'] = pd.to_numeric(sig_plot['regime_int'], errors='coerce')
        axr2.step(sig_plot['timestamp'], sig_plot['regime_step'], where='post', linewidth=2)
        axr2.set_ylim(-0.1, 2.1)
        axr2.set_ylabel('Regime (hourly)')
        axr2.set_title('Regime hour-by-hour')
        axr2.set_yticks([0, 1, 2])
        axr2.set_yticklabels(['Expansion', 'Slowdown', 'Recession'])
        axr2.grid(alpha=0.2)
    else:
        axr2.axis('off')
        axr2.text(0.5, 0.5, 'No signal history available.', ha='center', va='center')

    plt.tight_layout()
    plt.show()
else:
    print('No regime history available from live feed.')


displayHTML('<h2>4. Current feature state</h2>')
if feature_history.empty:
    print('No feature history available in silver.')
else:
    latest = feature_history.tail(1).iloc[0]
    prev = feature_history.tail(2).iloc[0] if len(feature_history) > 1 else pd.Series(dtype=float)
    rows = []
    for feature in FEATURES:
        if feature not in feature_history.columns:
            continue
        cur = _to_float(latest.get(feature))
        p = _to_float(prev.get(feature)) if not prev.empty else np.nan
        raw_key = f'{feature}_raw'
        raw_value = latest.get(raw_key) if raw_key in latest else np.nan
        rows.append((
            feature,
            FEATURE_LABELS.get(feature, feature),
            _to_float(importance_map.get(feature, np.nan)),
            _to_float(cur),
            _to_float(raw_value),
            cur - p if not pd.isna(cur) and not pd.isna(p) else np.nan,
            abs(_to_float(importance_map.get(feature, np.nan)) * (cur - p) if not pd.isna(cur) and not pd.isna(p) and not pd.isna(importance_map.get(feature, np.nan)) else np.nan),
        ))

    current_features = pd.DataFrame(rows, columns=['feature', 'label', 'weight', 'z_score', 'raw_value', 'delta_z', 'weighted_move'])
    current_features['weight_pct'] = current_features['weight'] / current_features['weight'].max() if not current_features.empty and current_features['weight'].max() not in (0, np.nan) else np.nan
    display(current_features.sort_values('weight', ascending=False).head(TOP_FEATURES).drop(columns=['feature']))
    display(importance_frame[['label', 'importance', 'importance_pct']].head(TOP_FEATURES).style.format({'importance': '{:.4f}', 'importance_pct': '{:.2%}'}))


displayHTML('<h2>5. Historical feature monitoring</h2>')
if feature_history.empty:
    print('No feature monitoring data available.')
else:
    top_for_chart = current_features.head(TOP_FEATURES)['feature'].tolist()
    if 'Price_Mom_24H' not in top_for_chart and 'Price_Mom_24H' in feature_history.columns:
        top_for_chart.append('Price_Mom_24H')

    panel_features = [f for f in top_for_chart if f in feature_history.columns and f != 'Price_Mom_24H']
    n_panels = max(1, len(panel_features))
    fig3, axes = plt.subplots(max(1, min(n_panels, 4)), 1, figsize=(16, 3 * max(1, min(n_panels, 4))), sharex=True)
    if n_panels == 1:
        axes = [axes]
    selected_window = feature_history.tail(24 * FEATURE_PLOT_DAYS)[['timestamp'] + panel_features].copy()
    for axis, feature in zip(axes, panel_features[:len(axes)]):
        series = pd.to_numeric(selected_window[feature], errors='coerce')
        axis.plot(selected_window['timestamp'], series, linewidth=1.5, label=FEATURE_LABELS.get(feature, feature))
        axis.set_ylabel('z-score')
        axis.axhline(0, color='black', linestyle=':', alpha=0.5)
        axis.set_title(FEATURE_LABELS.get(feature, feature))
        axis.grid(alpha=0.3)
        axis.legend(loc='upper left')

    if len(panel_features) > 4:
        extra_axes = plt.subplots(1, 1)
        extra_axes[1].axis('off')
    plt.tight_layout()
    plt.show()

    if 'Price_Mom_24H' in feature_history.columns:
        mom = feature_history[['timestamp', 'Price_Mom_24H']].dropna()
        if not mom.empty:
            mom_window = mom.tail(24 * FEATURE_PLOT_DAYS)
            fig4, ax4 = plt.subplots(figsize=(16, 3.5))
            bars = np.where(mom_window['Price_Mom_24H'] >= 0, 1, -1)
            ax4.bar(mom_window['timestamp'], mom_window['Price_Mom_24H'], color=['#2E8B57' if v >= 0 else '#C0392B' for v in mom_window['Price_Mom_24H']], width=0.03, alpha=0.65)
            ax4.set_title('24H momentum (dedicated panel)')
            ax4.set_ylabel('Price_Mom_24H')
            ax4.axhline(0, color='black', linestyle='--', alpha=0.5)
            ax4.grid(alpha=0.3)
            plt.show()


displayHTML('<h2>6. Attribution: Top drivers now</h2>')
if feature_history.empty:
    print('No attribution history available.')
else:
    feature_history['timestamp'] = pd.to_datetime(feature_history['timestamp'], errors='coerce')
    latest_row_features = feature_history.sort_values('timestamp').iloc[-1]
    previous_row_features = feature_history.sort_values('timestamp').iloc[-2] if len(feature_history) > 1 else None
    attribution_rows = []
    for feature in current_features['feature'].tolist():
        if feature not in feature_history.columns:
            continue
        cur = _to_float(latest_row_features.get(feature))
        prev = _to_float(previous_row_features.get(feature)) if previous_row_features is not None else np.nan
        delta = np.nan if (pd.isna(cur) or pd.isna(prev)) else cur - prev
        weight = _to_float(importance_map.get(feature, np.nan))
        attribution_rows.append((feature, delta, weight, abs(_to_float(delta) * weight) if not pd.isna(delta) and not pd.isna(weight) else np.nan, _to_float(delta)))

    attribution = pd.DataFrame(attribution_rows, columns=['feature', 'delta', 'weight', 'score', 'delta_abs'])
    if attribution.empty:
        print('Not enough history to compute feature deltas.')
    else:
        attribution = attribution.dropna(subset=['delta']).sort_values('score', ascending=False)
        attribution['feature'] = attribution['feature'].map(FEATURE_LABELS)
        attribution_top = attribution.head(8).reset_index(drop=True)
        attribution_top['driver_comment'] = attribution_top['feature'] + ': ' + attribution_top['delta'].map(lambda x: f'{x:+.3f}')
        display(attribution_top[['feature', 'delta', 'weight', 'score', 'driver_comment']].style.format({'delta': '{:+.3f}', 'weight': '{:.2%}', 'score': '{:.4f}'}))


displayHTML('<h2>7. Historical signals and trades</h2>')
sig_cols = [
    'timestamp', 'regime_name', 'regime_int', 'position'
]
if not signal_history.empty:
    available_sig = [c for c in sig_cols if c in signal_history.columns]
    print('Latest 24 hourly signals:')
    display(signal_history[available_sig].sort_values('timestamp', ascending=False).head(24))

if trade_history.empty:
    print('No trade history available in window.')
else:
    trade_cols = ['trade_ts', 'signal_ts', 'regime_name', 'target_position', 'switched', 'target_ticker', 'account_balance', 'account_equity', 'hourly_pnl', 'hourly_pnl_pct', 'day_pnl', 'day_pnl_pct', 'execution_mode']
    available_cols = [c for c in trade_cols if c in trade_history.columns]
    display(trade_history[available_cols].sort_values('trade_ts', ascending=False).head(100).style.format({
        'account_balance': '{:,.2f}', 'account_equity': '{:,.2f}', 'hourly_pnl': '{:+,.2f}', 'hourly_pnl_pct': '{:+.2%}',
        'day_pnl': '{:+,.2f}', 'day_pnl_pct': '{:+.2%}'
    }))


displayHTML('<h2>8. Execution / health / freshness</h2>')
health_rows = [
    ('Dashboard freshness (min)', fmt_minutes(latest_feed.get('dashboard_data_freshness_minutes'))),
    ('Audit lag (min)', fmt_minutes(latest_health.get('audit_lag_minutes'))),
    ('Latest audit ts', fmt_timestamp(latest_feed.get('latest_audit_ts'))),
    ('Last execution action', _as_text(latest_feed.get('last_execution_action'))),
    ('Skip reason', _as_text(latest_feed.get('last_skip_reason'))),
    ('Error reason', _as_text(latest_feed.get('last_error_reason'))),
    ('Skip / 24h', _as_text(latest_feed.get('last_skip_count_24h', 0))),
    ('Error / 24h', _as_text(latest_feed.get('last_error_count_24h', 0))),
    ('Rate-limit / 24h', _as_text(latest_feed.get('last_rate_limit_count_24h', 0))),
]
display(pd.DataFrame(health_rows, columns=['Metric', 'Value']))

if not health_history.empty:
    health_tail = health_history[['rollup_ts', 'audit_lag_minutes', 'last_execution_action', 'last_skip_count_24h', 'last_error_count_24h', 'last_rate_limit_count_24h']].copy()
    health_tail['rollup_ts'] = health_tail['rollup_ts'].dt.strftime('%Y-%m-%d %H:%M')
    display(health_tail.tail(24))
